# Synthetic Control: Unified Pipeline — COVID-Inclusive Robustness Check (LM_AIOE + MS_SCORE, Employment + Wage)

**Robustness variant:** Includes OEWS years 2020 and 2021 (COVID years) in the estimation sample.

Baseline (`synthetic_control_unified.ipynb`) excluded 2020–2021. This version includes them to test whether COVID volatility substantially changes the estimated treatment effect.

Treatment timing remains: 2022 = final pre-treatment year, 2023 = first post-treatment year (matching DID and GSC specifications).

Consolidates what used to be 4 separate notebooks (`synthetic_control_AIOE.ipynb`, `sc_wage.ipynb`, `synthetic_control_MS.ipynb`, `synthetic_control_wage_MS.ipynb`) into one file, so there is only ever **one** copy of `weighted_mean`, `run_sc`, and the donor-pool-shrinking logic — no more risk of fixing a bug in one file and forgetting the other three.

New in this version: the donor pool size **K is chosen by cross-validation** for each specification, instead of being fixed at an arbitrary value (previously K=20 everywhere).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.optimize import minimize
import warnings
warnings.filterwarnings("ignore")

## STEP 1: Load data
Tries the MS panel first (it should already contain both `LM_AIOE` and `MS_SCORE` columns, since it was built by merging MS_SCORE onto the harmonised LM_AIOE panel). Falls back to loading and merging both files if that's not the case — check the printed column list to confirm which path was taken.

In [ ]:
def load_combined_panel():
    try:
        df = pd.read_csv('bls_onet_felten_ms_panel_harmonised.csv')
        if 'LM_AIOE' in df.columns and 'MS_SCORE' in df.columns:
            print('Loaded single combined panel: bls_onet_felten_ms_panel_harmonised.csv')
            print(f'Columns available: {list(df.columns)}')
            return df
    except FileNotFoundError:
        pass

    print('Combined panel not found or missing a column — loading and merging both files instead.')
    df_aioe = pd.read_csv('bls_onet_felten_panel_harmonised.csv')
    df_ms   = pd.read_csv('bls_onet_felten_ms_panel_harmonised.csv')
    merge_keys = ['OCC_CODE', 'year']
    ms_cols = merge_keys + ['MS_SCORE']
    df = df_aioe.merge(df_ms[ms_cols].drop_duplicates(), on=merge_keys, how='left')
    print(f'Merged panel columns: {list(df.columns)}')
    return df

raw = load_combined_panel()
# COVID-INCLUSIVE: do not exclude years 2020, 2021
# raw = raw[~raw['year'].isin([2020, 2021])].copy()  # COMMENTED OUT for COVID-inclusive analysis
print(f'\nRows before any filtering: {len(raw)}')
print(f'Year range: {raw["year"].min()} to {raw["year"].max()}')

## STEP 2: Shared utility functions
One copy of each — used by every specification below.

In [ ]:
def weighted_mean(group, col, weight_col):
    """Weighted mean that excludes rows with a missing outcome, rather than
    filling missing values with 0 (the fillna(0) bug that previously caused
    extreme outliers when TOT_EMP for a missing-outcome row was large)."""
    valid = group[col].notna()
    w = group.loc[valid, weight_col].fillna(0)
    if w.sum() == 0:
        return np.nan
    return np.average(group.loc[valid, col], weights=w)


def build_occ_year_panel(raw_df, outcome_col, index_col):
    """Aggregate row-level data to occupation-year level for a given outcome
    ('log_emp' or 'log_wage') and exposure index ('LM_AIOE' or 'MS_SCORE')."""
    df = raw_df.copy()
    df['log_emp']  = np.log(df['TOT_EMP'].replace(0, np.nan))
    df['log_wage'] = np.log(df['A_MEDIAN'].replace(0, np.nan))

    if outcome_col == 'log_emp':
        occ_year = df.groupby(['OCC_CODE', 'year']).apply(
            lambda g: pd.Series({
                'OCC_TITLE': g['OCC_TITLE'].iloc[0],
                'log_emp':   np.log(g['TOT_EMP'].sum()),
                index_col:   g[index_col].iloc[0],
            })
        ).reset_index()
    else:  # log_wage
        occ_year = df.groupby(['OCC_CODE', 'year']).apply(
            lambda g: pd.Series({
                'OCC_TITLE': g['OCC_TITLE'].iloc[0],
                'log_wage':  weighted_mean(g, 'log_wage', 'TOT_EMP'),
                index_col:   g[index_col].iloc[0],
            })
        ).reset_index()

    occ_year = occ_year.dropna(subset=[index_col, outcome_col])
    return occ_year


def select_top_k_donors(treated_code, donor_codes, pivot, years, k):
    """Pick the k donors most similar to the treated unit over the given years,
    by Euclidean distance on the trajectory."""
    treated_vals = pivot.loc[years, treated_code].values
    distances = {}
    for donor in donor_codes:
        donor_vals = pivot.loc[years, donor].values
        distances[donor] = np.sqrt(np.sum((treated_vals - donor_vals) ** 2))
    ranked = sorted(distances.items(), key=lambda x: x[1])
    return [d[0] for d in ranked[:k]]


def fit_sc_weights(Y1, Y0):
    """Fit non-negative, sum-to-one synthetic control weights minimizing squared error."""
    n_donors = Y0.shape[1]

    def objective(w):
        return np.sum((Y1 - Y0 @ w) ** 2)

    constraints = {"type": "eq", "fun": lambda w: np.sum(w) - 1}
    bounds = [(0, 1)] * n_donors
    w0 = np.ones(n_donors) / n_donors
    result = minimize(objective, w0, method="SLSQP",
                      bounds=bounds, constraints=constraints,
                      options={"maxiter": 1000, "ftol": 1e-9})
    return result.x


def run_sc(treated_code, donor_codes, pivot, pre_years, all_years):
    """Run synthetic control for one treated occupation over the given donor pool."""
    Y1 = pivot.loc[all_years, treated_code].values
    Y0 = pivot.loc[all_years, donor_codes].values
    n_pre = len(pre_years)
    weights = fit_sc_weights(Y1[:n_pre], Y0[:n_pre, :])
    synthetic_full = Y0 @ weights
    gap = Y1 - synthetic_full
    pre_fit = np.sqrt(np.mean((Y1[:n_pre] - synthetic_full[:n_pre]) ** 2))
    return gap, weights, pre_fit, synthetic_full


print('Utility functions defined.')

## STEP 3: Cross-validation to select K (donor pool size)

Leave-one-pre-year-out: for each candidate K, hold out one pre-treatment year at a time, fit donor weights on the remaining pre-years (re-selecting the top-K donors using only the training years, to avoid leakage), predict the held-out year, and record the squared error. Average across held-out years and across all treated units, separately for each K, then pick the K with the lowest average error.

**Caveat worth keeping in mind:** with only 5 pre-treatment years, leaving one out leaves just 4 training points — so for K > 4 the training-step fit is already underdetermined before we even get to the held-out prediction. Treat the CV curve as informative, not as a precise optimum, and look at the plotted curve (not just the single minimum) before deciding.

In [ ]:
def cv_select_k(treated_codes, donor_codes, pivot, pre_years, k_grid):
    """Leave-one-pre-year-out CV for donor pool size K, averaged across all treated units.
    Returns (best_k, mean_cv_error_by_k, per_unit_errors_by_k)."""
    mean_error_by_k = {}
    all_errors_by_k = {k: [] for k in k_grid}

    for k in k_grid:
        for treated_code in treated_codes:
            sq_errors = []
            for held_out_year in pre_years:
                train_years = [y for y in pre_years if y != held_out_year]
                if len(train_years) < 2:
                    continue
                # Re-select top-k donors using only the training years (no leakage from held-out year)
                top_k = select_top_k_donors(treated_code, donor_codes, pivot, train_years, k)

                Y1_train = pivot.loc[train_years, treated_code].values
                Y0_train = pivot.loc[train_years, top_k].values
                w = fit_sc_weights(Y1_train, Y0_train)

                Y0_held = pivot.loc[held_out_year, top_k].values
                pred = Y0_held @ w
                actual = pivot.loc[held_out_year, treated_code]
                sq_errors.append((actual - pred) ** 2)

            if sq_errors:
                all_errors_by_k[k].append(np.mean(sq_errors))

        mean_error_by_k[k] = np.mean(all_errors_by_k[k]) if all_errors_by_k[k] else np.nan

    best_k = min(mean_error_by_k, key=mean_error_by_k.get)
    return best_k, mean_error_by_k, all_errors_by_k


def plot_cv_curve(mean_error_by_k, title, filename):
    fig, ax = plt.subplots(figsize=(8, 5))
    ks = sorted(mean_error_by_k.keys())
    errs = [mean_error_by_k[k] for k in ks]
    ax.plot(ks, errs, 'o-', color='#2980B9', linewidth=2)
    best_k = min(mean_error_by_k, key=mean_error_by_k.get)
    ax.axvline(best_k, color='#C0392B', linestyle='--', label=f'Selected K = {best_k}')
    ax.set_xlabel('K (donor pool size)')
    ax.set_ylabel('Mean leave-one-year-out squared error')
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.show()
    print(f'Saved {filename}')


print('CV functions defined. K will be selected separately for each of the 4 specifications below.')

## STEP 4: Run all four specifications
LM_AIOE × employment, LM_AIOE × wage, MS_SCORE × employment, MS_SCORE × wage — each goes through the same pipeline: build panel → split treated/donor → pivot → **CV-select K** → run SC for every treated occupation with its own top-K donors → average gap plot → regression on the index.

This will take a few minutes per specification, mostly spent in the CV step (it fits SC weights `len(k_grid) × len(treated_occs) × 5` times).

In [ ]:
K_GRID = [3, 5, 8, 12, 20, 30, 50, 75, 100, 150, 200]  # candidates to try; adjusted for COVID-inclusive sample

SPECS = [
    {'index': 'LM_AIOE',  'outcome': 'log_emp',  'label': 'aioe_emp'},
    {'index': 'LM_AIOE',  'outcome': 'log_wage', 'label': 'aioe_wage'},
    {'index': 'MS_SCORE', 'outcome': 'log_emp',  'label': 'ms_emp'},
    {'index': 'MS_SCORE', 'outcome': 'log_wage', 'label': 'ms_wage'},
]

all_spec_results = {}  # populated as we go, used for the final summary table

for spec in SPECS:
    index_col   = spec['index']
    outcome_col = spec['outcome']
    label       = spec['label']
    print('\n' + '=' * 70)
    print(f'SPEC: {index_col} x {outcome_col}  ({label})')
    print('=' * 70)

    # --- build panel ---
    occ_year = build_occ_year_panel(raw, outcome_col, index_col)
    print(f'Occupations with non-missing {index_col} and {outcome_col}: {occ_year["OCC_CODE"].nunique()}')
    print(f'Year range in occ_year: {occ_year["year"].min()} to {occ_year["year"].max()}')

    # --- split treated / donor ---
    idx_by_occ = occ_year.groupby('OCC_CODE')[index_col].first()
    median_idx = idx_by_occ.median()
    high_occs = idx_by_occ[idx_by_occ >= median_idx].index.tolist()
    low_occs  = idx_by_occ[idx_by_occ <  median_idx].index.tolist()

    # --- pivot ---
    pivot = occ_year.pivot(index='year', columns='OCC_CODE', values=outcome_col)
    pivot = pivot.replace([np.inf, -np.inf], np.nan).dropna(axis=1)
    available = set(pivot.columns)
    high_occs = [o for o in high_occs if o in available]
    low_occs  = [o for o in low_occs  if o in available]
    print(f'Treated: {len(high_occs)}, Donor pool: {len(low_occs)}')
    print(f'Years in pivot: {sorted(pivot.index.tolist())}')

    pre_years  = [y for y in pivot.index if y <= 2022]  # 2012-2022 with COVID-inclusive
    post_years = [y for y in pivot.index if y >= 2023]  # 2023+ post-treatment
    all_years  = pre_years + post_years
    print(f'Pre-treatment years: {pre_years}')
    print(f'Post-treatment years: {post_years}')

    # --- CV-select K (using a random subsample of treated units to keep runtime reasonable) ---
    cv_sample_size = min(40, len(high_occs))
    rng = np.random.default_rng(42)
    cv_sample = list(rng.choice(high_occs, size=cv_sample_size, replace=False))
    print(f'Running CV on a subsample of {cv_sample_size} treated occupations...')

    best_k, mean_err_by_k, _ = cv_select_k(cv_sample, low_occs, pivot, pre_years, K_GRID)
    print(f'CV-selected K for {label}: {best_k}')
    print('Mean CV error by K:', {k: round(v, 5) for k, v in mean_err_by_k.items()})
    plot_cv_curve(mean_err_by_k, f'CV error by K — {index_col} x {outcome_col} (COVID-INCLUSIVE)', f'cv_curve_{label}_covid.png')

    # --- run SC for every treated occupation using the CV-selected K ---
    results = {}
    for occ in high_occs:
        try:
            top_donors = select_top_k_donors(occ, low_occs, pivot, pre_years, best_k)
            gap, weights, pre_fit, synthetic = run_sc(occ, top_donors, pivot, pre_years, all_years)
            results[occ] = {'gap': gap, 'pre_fit': pre_fit, index_col: idx_by_occ[occ]}
        except Exception:
            pass

    pre_fits = [v['pre_fit'] for v in results.values()]
    print(f'Completed: {len(results)} occupations. Pre-period RMSE — mean: {np.mean(pre_fits):.4f}, median: {np.median(pre_fits):.4f}')

    # --- average gap plot ---
    gap_matrix = pd.DataFrame({occ: r['gap'] for occ, r in results.items()}, index=all_years)
    avg_gap = gap_matrix.mean(axis=1)
    se_gap  = gap_matrix.std(axis=1) / np.sqrt(len(results))

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(all_years, avg_gap, 'o-', color='#C0392B', linewidth=2, label='Average gap')
    ax.fill_between(all_years, avg_gap - 1.96 * se_gap, avg_gap + 1.96 * se_gap, alpha=0.2, color='#C0392B', label='95% CI')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.axvline(2021.5, color='gray', linewidth=1.5, linestyle='--', label='ChatGPT Launch')
    ax.set_xlabel('Year')
    ax.set_ylabel(f'Gap (Actual - Synthetic) in {outcome_col}')
    ax.set_title(f'Average SC Gap: {index_col} x {outcome_col} (CV-selected K={best_k}, COVID-INCLUSIVE)')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'sc_average_gap_{label}_covid.png', dpi=150)
    plt.show()
    print(f'Saved sc_average_gap_{label}_covid.png')

    # --- regression on the index ---
    post_idx = [all_years.index(y) for y in post_years]
    reg_data = pd.DataFrame({
        'OCC_CODE':     list(results.keys()),
        index_col:      [results[o][index_col] for o in results],
        'avg_post_gap': [results[o]['gap'][post_idx].mean() for o in results],
        'pre_fit':      [results[o]['pre_fit'] for o in results],
    })
    reg_data = reg_data.replace([np.inf, -np.inf], np.nan).dropna()

    X = sm.add_constant(reg_data[index_col])
    y = reg_data['avg_post_gap']
    model = sm.OLS(y, X).fit()  # standard SE — HC3 can be unstable with small samples
    print(f'\nOLS: avg_post_gap ~ {index_col}  (n={len(reg_data)})')
    print(f'  coef={model.params[index_col]:.4f}, p={model.pvalues[index_col]:.4f}')

    all_spec_results[label] = {
        'index': index_col, 'outcome': outcome_col, 'best_k': best_k,
        'n_treated': len(results), 'pre_fit_mean': np.mean(pre_fits), 'pre_fit_median': np.median(pre_fits),
        'coef': model.params[index_col], 'p_value': model.pvalues[index_col], 'n_reg': len(reg_data),
    }

print('\n\nAll four specifications complete.')

## STEP 5: Summary table across all four specifications

In [ ]:
summary = pd.DataFrame(all_spec_results).T
summary = summary[['index', 'outcome', 'best_k', 'n_treated', 'pre_fit_mean', 'pre_fit_median', 'coef', 'p_value', 'n_reg']]
print(summary.to_string())
summary.to_csv('sc_unified_summary_covid.csv', index=False)
print('\nSaved sc_unified_summary_covid.csv')